# Post-process

Run from the directory containing `aztc_full_localdoc`. Reads all `train-*-of-*.parquet` shards together and writes processed Parquet shards into `processed_data/`.

The output shard count is calculated from the remaining row count, using the first source shard’s row count as the target. Files follow the `train-00000-of-000NN.parquet` naming pattern, with the suffix reflecting the actual output shard count.

Rerunning replaces matching train shards in `processed_data/`. Original input shards, unrelated files, original columns, and retained documents’ boilerplate remain unchanged.

Near-duplicate features and pairs are saved under `near_duplicated_data_post/`. Detection uses the EDA’s approximate MinHash method: binary hashed 5-word shingles, three hash tables, and similarity ≥90%. Documents with fewer than 10 normalized words bypass near-duplicate detection. For each detected pair, the document with the larger text-hash ID is removed; chains may lose multiple members.

Run the EDA first to install FastText and create `/tmp/lid.176.bin`. Each Spark partition loads the existing local model once.

Printed counts show rows remaining after each filtering step. Near-duplicate matches are counted as pairs, not distinct documents removed.

In [1]:
import os
import shutil
from pathlib import Path
import fasttext
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.feature import HashingTF, MinHashLSH

os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 10g pyspark-shell"
spark = SparkSession.builder.appName("post-process").master("local[*]").getOrCreate()
BASE = "near_duplicated_data_post"
OUTPUT = "processed_data"

def counted(data, label, previous=None):
    data = data.persist()
    print(f"{label}: {data.count():,} rows")
    if previous is not None:
        previous.unpersist()
    return data

df = counted(spark.read.parquet("aztc_full_localdoc/train-*-of-*.parquet"), "Input dataset")

Input dataset: 4,404,338 rows


## Filter and remove exact duplicates
The allowed-character filter also removes Cyrillic. Exact deduplication precedes language detection to avoid repeated inference.

In [2]:
for label, condition in [
    ("After ANL filter", F.col("source") == "ANL"),
    ("After null/empty-text removal", F.col("text").isNotNull() & (F.col("text") != "")),
    ("After whitespace-only removal", ~F.col("text").rlike(r"(?U)^\s+$")),
    ("After character filtering (including Cyrillic)", ~F.col("text").rlike(r"[^\p{IsLatin}\p{N}\p{P} \t\r\n\f]")),
    ("After minimum-length filter", F.length("text") >= 500),
]:
    df = counted(df.filter(condition), label, df)

df = counted(df.dropDuplicates(["text"]), "After exact-duplicate removal", df)

After ANL filter: 240,655 rows
After null/empty-text removal: 240,655 rows
After whitespace-only removal: 240,655 rows
After character filtering (including Cyrillic): 235,686 rows
After minimum-length filter: 225,594 rows
After exact-duplicate removal: 211,275 rows


## Keep Azerbaijani
Run the EDA first to create `/tmp/lid.176.bin`. Each Spark partition loads the existing local model once and retains only texts classified as Azerbaijani.

In [3]:
model_path = Path("/tmp/lid.176.bin")

def keep_azerbaijani(rows):
    model = fasttext.load_model(str(model_path))
    for row in rows:
        if model.predict(" ".join(row.text.split()))[0][0] == "__label__az":
            yield row

df = counted(spark.createDataFrame(df.rdd.mapPartitions(keep_azerbaijani), df.schema), "After Azerbaijani language filter", df)

After Azerbaijani language filter: 211,275 rows


## Persist near-duplicate features and pairs

In [4]:
columns = df.columns
docs = df.withColumn("_post_id", F.sha2("text", 256))
words = docs.select("_post_id", "text").withColumn(
    "words", F.split(F.trim(F.regexp_replace(F.lower("text"), r"[^\p{L}\p{N}]+", " ")), r"\s+")
).filter(F.size("words") >= 10)
shingles = words.withColumn("shingles", F.expr(
    "transform(sequence(1, size(words) - 4), i -> array_join(slice(words, i, 5), ' '))"
))
features = HashingTF(inputCol="shingles", outputCol="features", numFeatures=1 << 20, binary=True).transform(shingles)
features.select(F.col("_post_id").alias("id"), "text", "features").write.mode("overwrite").parquet(f"{BASE}/features")
vectors = spark.read.parquet(f"{BASE}/features").select("id", "features")
print(f"Eligible for near-duplicate matching: {vectors.count():,} rows")

model = MinHashLSH(inputCol="features", outputCol="hashes", numHashTables=3, seed=42).fit(vectors)
vectors = model.transform(vectors).persist()
pairs = (
    model.approxSimilarityJoin(vectors, vectors, 0.1000001, distCol="distance")
    .filter(F.col("datasetA.id") < F.col("datasetB.id"))
    .select(F.col("datasetA.id").alias("id_a"), F.col("datasetB.id").alias("id_b"),
            (1 - F.col("distance")).alias("similarity"))
    .filter(F.col("similarity") >= 0.90)
)
pairs.write.mode("overwrite").parquet(f"{BASE}/pairs")
vectors.unpersist()
pairs = spark.read.parquet(f"{BASE}/pairs")
print(f"Near-duplicate matches: {pairs.count():,} pairs")

Eligible for near-duplicate matching: 211,275 rows
Near-duplicate matches: 3,454 pairs


## Remove detected near duplicates and write processed shards

In [5]:
removed = pairs.select(F.col("id_b").alias("_post_id")).distinct()
processed = docs.join(removed, "_post_id", "left_anti").select(*columns).persist()
row_count = processed.count()
print(f"After near-duplicate removal: {row_count:,} rows")
df.unpersist()

source_shard = min(Path("aztc_full_localdoc").glob("train-*-of-*.parquet"))
rows_per_shard = spark.read.parquet(str(source_shard)).count()
shard_count = max(1, (row_count + rows_per_shard - 1) // rows_per_shard)

staging = Path(OUTPUT) / "_spark_shards"
processed.repartition(shard_count).write.mode("overwrite").parquet(str(staging))
processed.unpersist()

shards = sorted(staging.glob("part-*.parquet"))
for old in Path(OUTPUT).glob("train-*-of-*.parquet"):
    old.unlink()
for i, shard in enumerate(shards):
    shard.rename(Path(OUTPUT) / f"train-{i:05d}-of-{len(shards):05d}.parquet")
shutil.rmtree(staging)

print(f"Saved: {spark.read.parquet(f'{OUTPUT}/train-*-of-*.parquet').count():,} rows | Shards: {len(shards)} | Directory: {OUTPUT}")

After near-duplicate removal: 208,293 rows
Saved: 208,293 rows | Shards: 1 | Directory: processed_data
